# Amplitude encoding and qubit budgets
**Quantum Machine Learning · Week 1 (Data meets qubits) · Day 3**

Lesson page: https://sara-aminpour.com/tutorials/qml/w1d3.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

**Amplitude encoding** stores a whole vector in the amplitudes of a state: $2^n$ numbers in only $n$ qubits.
An 8 x 8 digit (64 pixels) fits in 6 qubits. Today: how, and what it really costs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import StatePreparation
from qiskit.quantum_info import Statevector
from sklearn.datasets import load_digits

digits = load_digits()
x = digits.data[0]                                  # a "0": 64 pixel values
amplitudes = x / np.linalg.norm(x)                  # a state must have length 1
state = Statevector(amplitudes)                     # 64 amplitudes = 6 qubits
print("qubits:", state.num_qubits)
print("probabilities = normalized pixels squared:", np.allclose(state.probabilities(), amplitudes ** 2))

## The catch: preparing the state
A circuit has to *make* that state from $|000000\rangle$. For an arbitrary vector, the number of CNOTs grows
like $2^n$: the "input problem" of quantum machine learning.

In [ ]:
rng = np.random.default_rng(0)
ns, cnots = range(1, 9), []
for n in ns:
    v = rng.normal(size=2 ** n)
    qc = QuantumCircuit(n)
    qc.append(StatePreparation(v / np.linalg.norm(v)), range(n))
    cnots.append(transpile(qc, basis_gates=["cx", "u"], optimization_level=1).count_ops().get("cx", 0))
    print(f"{n} qubit{'s' if n > 1 else ' '} ({2 ** n:3d} numbers): {cnots[-1]:3d} CNOTs")

plt.semilogy(list(ns), np.maximum(cnots, 0.8), "o-", color="#ff7a59", label="CNOTs to prepare the state")
plt.semilogy(list(ns), 2.0 ** np.array(ns), "--", color="#8a85a3", label=r"$2^n$")
plt.xlabel("qubits n")
plt.legend()
plt.show()

## The other catch: reading it back
Each shot reveals a single basis state. Estimating 64 amplitudes takes many shots.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(8, 2.3), layout="constrained")
axes[0].imshow(x.reshape(8, 8), cmap="Purples")
axes[0].set_title("original")
for ax, shots in zip(axes[1:], [100, 1000, 100000]):
    counts = np.bincount(rng.choice(64, size=shots, p=amplitudes ** 2), minlength=64)
    estimate = np.sqrt(counts / shots)
    ax.imshow(estimate.reshape(8, 8), cmap="Purples")
    ax.set_title(f"{shots:,} shots, F = {np.dot(estimate, amplitudes) ** 2:.3f}")
for ax in axes:
    ax.axis("off")
plt.show()

## A qubit budget
Today's simulators handle about 30 qubits, and hardware is noisy. A common recipe: shrink the data with
PCA (ML week 6), then angle-encode the few components that are left.

In [ ]:
from sklearn.decomposition import PCA

for k in [4, 6, 8]:
    pca = PCA(n_components=k).fit(digits.data)
    print(f"PCA to {k} components keeps {pca.explained_variance_ratio_.sum():.0%} of the variance -> {k} qubits with angle encoding")
print("for comparison: 64 qubits for angle encoding of all pixels, or 6 qubits and ~57 CNOTs for amplitude encoding")

## Your turn: what amplitude encoding forgets
Normalizing throws away a vector's length. Amplitude-encode the iris flowers (4 features, 2 qubits) and
show that a flower and a "double-size" copy of it (every feature times 2) give exactly the same state.
Then compute the fidelity between two flowers of different species.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
flower = iris.data[0]
def amp_state(v):
    return None          # TODO: Statevector(v / np.linalg.norm(v))
# TODO: compare amp_state(flower) and amp_state(2 * flower) with state_fidelity, then a setosa (row 0) and a virginica (row 100)

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
from sklearn.datasets import load_iris
from qiskit.quantum_info import state_fidelity

iris = load_iris()
flower = iris.data[0]
def amp_state(v):
    return Statevector(v / np.linalg.norm(v))
print(f"a flower and its double: fidelity {state_fidelity(amp_state(flower), amp_state(2 * flower)):.4f}")
print(f"setosa vs virginica:     fidelity {state_fidelity(amp_state(iris.data[0]), amp_state(iris.data[100])):.4f}")
print("The size of the flower is gone; only its proportions remain. Very different flowers can still look")
print("alike, because all four iris features are positive: every state sits in one corner of the sphere.")